# Opening and consolidating your data

Instruments rarely produce one file. They produce thousands, each holding a
short slice of the acquisition — often several fibers at once, and several
acquisitions on top of that. Xdas links them into one *virtual* dataset that
you slice as if it were a single array, without copying a byte.

It all goes through `xd.open`, which dispatches on the shape of what you hand
it: a path, a list of paths, a wildcard, or a directory pattern. We take those
four in turn, and end on the master files every later notebook opens.

In [1]:
import xdas as xd

The data is ten minutes of the ABYSS experiment: two interrogators, three
fibers between them, one file every ten seconds.

In [2]:
%ls data/gps_multicable/*
%ls data/gps_multicable/SER/N | head -3

data/gps_multicable/CCN:
N/

data/gps_multicable/SER:
N/  S/


122001.hdf5
122011.hdf5
122021.hdf5


## One file

Hand `xd.open` a plain path and you get a single `DataArray`. The file format
is worked out from the file itself, so there is nothing to declare — pass
`engine` only to force a reader that cannot be recognized.

In [3]:
xd.open("data/gps_multicable/SER/N/122001.hdf5")

<xdas.DataArray (time: 625, distance: 10000)>
VirtualSource: 11.9MB (int16)
Coordinates:
  * time (time): 2023-11-03T12:20:01.000 to 2023-11-03T12:20:10.984
  * distance (distance): 0.000 to 153179.709

Ten seconds of 10 000 channels. `VirtualSource` is the important word in
that repr: nothing has been read but the metadata, and `data` is a pointer
into the file rather than an array in memory. Samples are fetched only when
you ask for values.

## A few files

Pass a list and the files are linked along their first dimension — here
time — into a single array:

In [4]:
xd.open(
    [
        "data/gps_multicable/SER/N/122001.hdf5",
        "data/gps_multicable/SER/N/122011.hdf5",
        "data/gps_multicable/SER/N/122021.hdf5",
    ]
)

<xdas.DataArray (time: 1875, distance: 10000)>
VirtualStack: 35.8MB (int16)
Coordinates:
  * distance (distance): 0.000 to 153179.709
  * time (time): 2023-11-03T12:20:01.000 to 2023-11-03T12:20:30.984

Thirty seconds now, and a `VirtualStack` rather than a `VirtualSource`: one
pointer per file, stacked. `dim` picks the axis to link along if it is not
the first one.

The coordinates are checked as the files are joined. These were
GPS-synchronized and abut to the sample, so nothing needs saying; when the
timestamps are less trustworthy, `tolerance` decides how much slack counts as
continuous. Notebook 04 takes that apart.

## A whole acquisition

Listing files by hand does not scale. A wildcard — `*`, `?` or `[…]` — opens
everything that matches:

In [5]:
da = xd.open("data/gps_multicable/SER/N/*.hdf5")
da

<xdas.DataArray (time: 37500, distance: 10000)>
VirtualStack: 715.3MB (int16)
Coordinates:
  * distance (distance): 0.000 to 153179.709
  * time (time): 2023-11-03T12:20:01.000 to 2023-11-03T12:30:00.984

Sixty files, ten minutes, one array. The cost of opening is one metadata
read per file, so this stays cheap into the thousands; past that, `parallel`
spreads the scan over several processes and `verbose=True` gives you a
progress bar.

From here on the fact that there were ever sixty files is gone. Selection
works across the whole set as if it were one array — `slice` is required, as
in xarray:

In [6]:
da.sel(
    time=slice("2023-11-03T12:26:40", "2023-11-03T12:27:50"),
    distance=slice(20_000, 100_000),
)

<xdas.DataArray (time: 4375, distance: 5222)>
VirtualStack: 43.6MB (int16)
Coordinates:
  * distance (distance): 20007.271 to 99990.395
  * time (time): 2023-11-03T12:26:40.008 to 2023-11-03T12:27:49.992

## A whole experiment

One `DataArray` describes one array. As soon as you have several — three
fibers interrogated at the same time, or one acquisition whose parameters
changed halfway — you need a container that can hold arrays that do not
share a shape. That is `DataCollection`: a tree whose nodes are either a
`DataMapping` (a `dict`) or a `DataSequence` (a `list`), and both behave
exactly like the Python type they are named after.

The files are organized in folders: the node where the interrogator sits,
then the fiber direction (`N` north, `S` south), then the files themselves.
Braces name a folder level and brackets name the files, so `xd.open` maps
that layout straight onto the tree:

In [7]:
dc = xd.open("data/gps_multicable/{node}/{cable}/[record].hdf5")
dc

<xdas.DataCollection: 3 leaves, 2.1 GB>
node  cable  record
CCN   N           0  (time: 37500, distance: 9998)   715.1 MB
SER   N           0  (time: 37500, distance: 10000)  715.3 MB
      S           0  (time: 37500, distance: 10000)  715.3 MB

The CCN node has one fiber going north; SER has two, north and south. Each
fiber holds a single acquisition here, because no parameter changed over
these ten minutes — otherwise the `record` list would have more than
one entry.

`plot_availability` is the quickest way to check you opened what you think
you opened. Each leaf of the tree gets its own timeline:

In [8]:
xd.plot_availability(dc, dim="time", height=300)

### Reaching into the tree

Indexing works level by level, as with any nesting of dicts and lists:

In [9]:
dc["SER"]["N"][0]

<xdas.DataArray (time: 37500, distance: 10000)>
VirtualStack: 715.3MB (int16)
Coordinates:
  * distance (distance): 0.000 to 153179.709
  * time (time): 2023-11-03T12:20:01.000 to 2023-11-03T12:30:00.984

`query` is more convenient once the tree gets deep. It addresses the levels
by name rather than by position, and matches with wildcards:

In [10]:
dc.query(node="S*")

<xdas.DataCollection: 2 leaves, 1.4 GB>
node  cable  record
SER   N           0  (time: 37500, distance: 10000)  715.3 MB
      S           0  (time: 37500, distance: 10000)  715.3 MB

Name several levels to walk all the way down to a single leaf:

In [11]:
dc.query(node="SER", cable="N")

<xdas.DataCollection: 1 leaf, 715.3 MB>
node  cable  record
SER   N           0  (time: 37500, distance: 10000)  715.3 MB

### Operations that map over the tree

`query` chooses *which* leaves are kept, by their position in the hierarchy.
`sel` trims *inside* each leaf, by coordinate value — and like the other
`DataArray` methods that make sense elementwise, it is applied to every leaf
at once:

In [12]:
dc.sel(time=slice("2023-11-03T12:26:40", "2023-11-03T12:27:50"))

<xdas.DataCollection: 3 leaves, 250.4 MB>
node  cable  record
CCN   N           0  (time: 4376, distance: 9998)   83.4 MB
SER   N           0  (time: 4375, distance: 10000)  83.4 MB
      S           0  (time: 4376, distance: 10000)  83.5 MB

## Cable geometry

One thing the interrogator's files do not carry: where each channel physically
is. That comes from the survey of the fiber's route — here the digitized cable
trace from the deployment paper, one `distance, latitude, longitude` row per
few metres, in `data/geometry/{node}_{cable}.csv`.

Attaching it now, before saving, means every notebook downstream gets it for
free instead of re-interpolating a CSV of its own:

In [13]:
import numpy as np
import pandas as pd


def get_cable_coords(da, node, cable):
    """Interpolate a cable's surveyed route onto the channels of `da`."""
    geometry = pd.read_csv(f"data/geometry/{node}_{cable}.csv")
    distance = geometry["distance"].to_numpy() * 1000  # km -> m, like `da`
    return tuple(
        np.interp(
            da["distance"].values,
            distance,
            geometry[column].to_numpy(),
            left=np.nan,
            right=np.nan,
        )
        for column in ("latitude", "longitude")
    )

Each leaf gets its own survey, so interpolate once per cable and attach the
result as two extra coordinates along `distance`:

In [14]:
for node in dc:
    for cable in dc[node]:
        da = dc[node][cable][0]
        latitude, longitude = get_cable_coords(da, node, cable)
        dc[node][cable][0] = da.assign_coords(
            latitude=("distance", latitude),
            longitude=("distance", longitude),
        )

dc["SER"]["N"][0]

<xdas.DataArray (time: 37500, distance: 10000)>
VirtualStack: 715.3MB (int16)
Coordinates:
  * distance (distance): 0.000 to 153179.709
  * time (time): 2023-11-03T12:20:01.000 to 2023-11-03T12:30:00.984
    latitude (distance): [nan ... nan]
    longitude (distance): [nan ... nan]

Two things stand out. `SER/N`'s surveyed route stops at 102 km, short of the
interrogator's 153 km range, so the last third of the channels get `NaN`
rather than a made-up position. And the first 4.6 km repeat a single
coordinate: that stretch sits coiled at the shore station, where consecutive
channels genuinely do not move in space.

In [15]:
lat = dc["SER"]["N"][0]["latitude"].values
print(f"{np.isnan(lat).sum()} of {len(lat)} SER/N channels have no mapped position")

3337 of 10000 SER/N channels have no mapped position


## Saving the link

The consolidated view is written to disk without copying the data: the file
holds pointers to where the samples actually live. These are the files you
share with the rest of the team, and the ones the following notebooks open.

In [16]:
dc.to_netcdf("outputs/multicable.nc")  # virtual=True by default
dc["SER"]["N"][0].to_netcdf("outputs/singlecable.nc")

Reading them back is the single-file form again — and the same call either
way, since Xdas works out on its own whether the file holds a collection or
an array:

In [17]:
xd.open("outputs/multicable.nc")

<xdas.DataCollection: 3 leaves, 2.1 GB>
node  cable  record
CCN   N           0  (time: 37500, distance: 9998)   715.1 MB
SER   N           0  (time: 37500, distance: 10000)  715.3 MB
      S           0  (time: 37500, distance: 10000)  715.3 MB

In [18]:
xd.open("outputs/singlecable.nc")

<xdas.DataArray (time: 37500, distance: 10000)>
VirtualSource: 715.3MB (int16)
Coordinates:
  * distance (distance): 0.000 to 153179.709
  * time (time): 2023-11-03T12:20:01.000 to 2023-11-03T12:30:00.984
    latitude (distance): [nan ... nan]
    longitude (distance): [nan ... nan]

## When the files do not agree

Files whose chunks have different shapes cannot go into one array. When you
open such a set, Xdas groups what is compatible and returns a sequence rather
than failing:

In [19]:
xd.open("data/gps_multiacq/*.hdf5")

<xdas.DataCollection: 3 leaves, 672.2 MB>
record
     0  (time: 6250, distance: 9998)   119.2 MB
     1  (time: 21000, distance: 6666)  267.0 MB
     2  (time: 6000, distance: 24994)  286.0 MB

Three acquisitions, so two parameter changes. Reading the sampling intervals
back confirms what changed and when:

In [20]:
print("#   dt      dx")
print("-----------------")
for acquisition, da in enumerate(xd.open("data/gps_multiacq/*.hdf5")):
    dt = xd.get_sampling_interval(da, "time")
    dx = xd.get_sampling_interval(da, "distance")
    print(f"{acquisition}   {dt * 1000:>2.0f} ms   {dx:>5.2f} m")

#   dt      dx
-----------------
0   16 ms   15.32 m
1   10 ms   15.32 m
2   10 ms    4.09 m


```{note}
A wildcard therefore returns whichever of the two fits: one `DataArray` when
everything is compatible, a `DataCollection` when it is not. Pass
`squeeze=False` if you would rather always get a collection, and never have
to branch on the type.
```